# 2.2 SIMT 与 SIMD 介绍
Ascend C 支持两种并行执行模型：**SIMD**（单指令多数据）与 **SIMT**（单指令多线程）。两者在核心思想、适用场景和编程方式上差异显著，是算子编程的关键选型依据。


## 一、SIMD 与 SIMT 对比
| 模型 | 全称 | 核心思想 | 通俗比喻 | 适用场景 |
|---|---|---|---|---|
| **SIMD** | 单指令多数据 | 一条指令同时操作多个同构数据 | 一个厨师同时翻炒多份相同的菜 | 规整、高密度计算（矩阵乘、卷积、逐元素运算） |
| **SIMT** | 单指令多线程 | 一条指令驱动多个独立线程 | 一个厨师指挥多个助手，各切各的菜 | 不规则访问、分支密集、稀疏计算（Gather、Scatter） |

**关键区别**：SIMD 要求数据**连续排列**，通过基地址+步长批量寻址；SIMT 的每个线程**独立寻址**，天然适配离散访存。


## 二、编程步骤对比
两种模型均遵循 SPMD（单程序多数据）模型，但数据搬运方式不同：

| 步骤 | SIMD 编程 | SIMT 编程 |
|---|---|---|
| ① 分块 | 将数据划分为均匀块，每个 AI Core 负责一块 | 建立线程索引与数据索引的一一对应 |
| ② 数据搬入 | **显式调用**搬运 API，从 GM 搬到本地存储 | 通过指针**直接访问** GM，硬件自动加载到寄存器 |
| ③ 计算 | 调用 **SIMD 指令**（向量/矩阵 API） | 编写**标量代码**（支持分支、循环） |
| ④ 数据搬出 | **显式调用**搬运 API，写回 GM | 通过指针**直接写回** GM |

> SIMD 需显式管理数据搬运与同步；SIMT 无需显式搬运，但多线程协作时需同步指令。


## 三、典型算子结构与硬件映射
不同算子的数据访问模式决定了应选用 SIMD 还是 SIMT：

| 算子结构 | 代表算子 | 数据流 | 计算单元 | 编程模型 | 本章样例 |
|---|---|---|---|---|---|
| 连续矢量 | Add、ReLU、Softmax | GM ↔ UB | Vector Core | SIMD | 2.5 C API Add |
| 矩阵 | MatMul、Conv2D | GM → L1 → L0A/L0B → L0C → GM | Cube Core | SIMD | 2.6 Tensor MatMul |
| 离散矢量 | Gather、Scatter | GM（直接寻址） | Vector Core | SIMT | 2.7 SIMT Gather |

- **连续矢量**：数据连续排列，SIMD 一次处理整段，如 $z[i] = x[i] + y[i]$。
- **矩阵**：矩阵乘加累加，Cube 单元执行 Mmad，搬运时自动完成布局转换。
- **离散矢量**：每个元素源地址由 index 决定，SIMT 线程独立寻址，如 `output[i] = input[index[i]]`。


## 四、选型指南
Ascend 950 采用**以 SIMD 为主、SIMT 为辅**的新同构编程模型：矩阵和向量计算中的 SIMD 部分配置超过 90% 算力，SIMT 作为灵活性补充。

| 编程模型 | 支持范围 | 适用场景 | 芯片支持 |
|---|---|---|---|
| SIMD（主） | 向量、矩阵、融合计算 | 规整、高密度任务 | 昇腾全系列 |
| SIMT（辅） | 仅向量计算 | 离散访问、复杂分支、稀疏计算 | 仅 Ascend 950 |
| SIMD + SIMT 混合 | 向量、矩阵、融合计算 | 用 SIMT 处理不规则逻辑，SIMD 批量计算 | 仅 Ascend 950 |

> **选择原则**：数据连续用 SIMD，访存离散用 SIMT。开发常规高性能算子优先选 SIMD；涉及离散数据或复杂分支时选 SIMT。

> 下一节：[2.3 对应典型算子结构](02.03_typical_operator_structure.ipynb)
